# 03 -- Evaluasi Formal, Kalibrasi, OOD, Grad-CAM & Ekspor ONNX (spec Sec7-8)

Notebook ini jalan SETELAH keempat backbone (Tiny/Small/Base/Large) selesai
training (`02_train.ipynb`). Untuk setiap model, notebook ini:

1. **Evaluasi formal** di test set held-out zero-leakage -- bootstrap 95% CI
   (2000 resample), threshold dipilih di VALIDATION (fix F4 -- eksperimen
   lama memilih threshold optimal di test set, itu kebocoran data), kurva
   ROC/PR, confusion matrix, risk-coverage curve (selective prediction).
2. **Kalibrasi** -- temperature scaling di-fit di validation, ECE +
   reliability diagram di test.
3. **Gerbang OOD** -- jarak Mahalanobis di ruang fitur GAP, di-fit di
   fitur training, dievaluasi terhadap campuran ID (test X-ray) vs OOD
   (CIFAR-10, dataset publik non-X-ray -- lihat keputusan di CLAUDE.md).
4. **Verifikasi Grad-CAM analitik** -- bandingkan heatmap dari turunan
   closed-form (`src/fracture/gradcam.py`, NumPy murni) vs ground-truth
   `tf.GradientTape` di Keras asli, KEDUA arah kelas (fractured/
   not_fractured -- fix B3), selisih absolut maksimum harus < 1e-4.
5. **Ekspor ONNX** -- model 2-output `[prob, featmap]` (backbone CNN
   saja) + bobot head (`Dense-512`, `Dense-1`) diekspor terpisah sebagai
   `.npz`, supaya Grad-CAM & OOD bisa dihitung NumPy murni di server
   tanpa TensorFlow (spec Sec8). Parity probabilitas ONNX vs Keras juga
   diverifikasi (< 1e-4).

## KONVENSI LABEL -- WAJIB DIPATUHI DI SELURUH CELL DI BAWAH

`flow_from_dataframe(class_mode="binary")` meng-assign index kelas
berdasar urutan ALFABETIS: **index 0 = "fractured", index 1 =
"not_fractured"**. Output sigmoid mentah model = P(not_fractured) --
TERBALIK dari framing klinis biasa (fractured = positif). Seluruh fungsi
di `src/fracture/{evaluate,calibration,gradcam,ood}.py` menerima
`prob_fractured`/`y_fractured` yang SUDAH dikonversi -- konversi
dilakukan eksplisit di notebook ini, bukan didiamkan implisit:

```python
prob_fractured = 1 - raw_sigmoid_output
y_fractured = (np.asarray(generator.classes) == 0).astype(int)  # index 0 = fractured; np.asarray karena .classes bisa list biasa tergantung versi Keras
```

## Yang BELUM ada di notebook ini

`clahe_ablation` (spec Sec11) sengaja TIDAK diisi di `results/metrics.json`
-- itu ablation study terpisah yang baru masuk akal setelah model terbaik
dari keempat backbone ini ditentukan. Field itu ditambahkan di iterasi
berikutnya, BUKAN diisi dengan angka rekaan di sini.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Konfigurasi -- samakan dengan yang dipakai `02_train.ipynb`

In [ ]:
DATASET_ROOT = "/content/drive/MyDrive/project-fracture/dataset/Bone_Fracture_Dataset"  # sama seperti 02_train.ipynb -- GANTI kalau strukturmu beda
REPO_URL = "https://github.com/alianhar/project-fracture.git"
REPO_DIR = "/content/project-fracture"
RUNS_ROOT = "/content/drive/MyDrive/fracture-runs"      # tempat best.keras keempat model (ditulis 02_train.ipynb)
EXPORT_ROOT = "/content/drive/MyDrive/fracture-exports"  # tempat .onnx + _head.npz keempat model (ditulis notebook ini)

BACKBONES = ["tiny", "small", "base", "large"]
N_BOOTSTRAP = 2000
OOD_SAMPLE_SIZE = 300  # jumlah gambar CIFAR-10 dipakai sbg referensi OOD

In [ ]:
!pip -q install pyyaml onnxruntime
!pip -q install -U tf2onnx  # -U: versi lama tf2onnx tidak punya handler untuk op Erfc (dipakai GELU eksak ConvNeXt)

In [ ]:
import os
import shutil

def _is_valid_git_repo(path):
    return os.path.isdir(os.path.join(path, ".git"))

if os.path.exists(REPO_DIR) and not _is_valid_git_repo(REPO_DIR):
    print(f"{REPO_DIR} ada tapi bukan git repo valid (sisa percobaan gagal) -- dihapus, clone ulang.")
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

assert _is_valid_git_repo(REPO_DIR), (
    f"Clone/pull gagal -- {REPO_DIR} bukan git repo valid. "
    "Cek repo GitHub public & REPO_URL benar (lihat output !git di atas)."
)

import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
import gc
import hashlib
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
import yaml
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from src.fracture.data import IMG_SIZE, manifest_to_dataframe, preprocess_image
from src.fracture import evaluate, calibration, gradcam, ood, export_onnx

Path(EXPORT_ROOT).mkdir(parents=True, exist_ok=True)

## Salin dataset ke disk lokal Colab (sekali per sesi)

Sama seperti `02_train.ipynb` -- baca gambar langsung dari Drive tiap
step itu bottleneck I/O parah (lihat CLAUDE.md, ~47x lebih lambat).

In [ ]:
import json as _json

LOCAL_DATASET_ROOT = "/content/dataset_local"
_sentinel = Path(LOCAL_DATASET_ROOT) / ".copy_done"

if not _sentinel.exists():
    with open(f"{REPO_DIR}/results/split_manifest.json") as f:
        _manifest = _json.load(f)

    n = len(_manifest["clusters"])
    print(f"Menyalin {n} gambar unik dari Drive ke disk lokal Colab (sekali saja per sesi)...")
    for i, c in enumerate(_manifest["clusters"]):
        src = Path(DATASET_ROOT) / c["canonical_path"]
        dst = Path(LOCAL_DATASET_ROOT) / c["canonical_path"]
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        if (i + 1) % 500 == 0:
            print(f"  {i + 1}/{n}")

    _sentinel.parent.mkdir(parents=True, exist_ok=True)
    _sentinel.touch()
    print("Selesai menyalin.")
else:
    print("Sudah pernah disalin di sesi ini -- lewati.")

## Generator evaluasi -- train/val/test TANPA augmentasi, shuffle=False

Beda dengan `make_generators()` di `src/fracture/data.py` (dipakai untuk
TRAINING, train_gen di situ shuffle=True + augmentasi): di sini KETIGA
split butuh urutan deterministik supaya `.classes` bisa dipasangkan
1:1 dengan output `.predict()` -- termasuk train (untuk fit Mahalanobis
OOD di fitur GAP, augmentasi/shuffle tidak relevan untuk itu).

In [ ]:
manifest_path = f"{REPO_DIR}/results/split_manifest.json"
BATCH_SIZE = 16  # sama dengan configs/base.yaml -- evaluasi tidak sensitif ke batch size, cuma pengaruh kecepatan

_df = manifest_to_dataframe(manifest_path, LOCAL_DATASET_ROOT)
_eval_datagen = ImageDataGenerator(preprocessing_function=preprocess_image)

def _make_eval_gen(split):
    return _eval_datagen.flow_from_dataframe(
        _df[_df["split"] == split], x_col="filename", y_col="class",
        target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode="binary", shuffle=False,
    )

train_gen_eval = _make_eval_gen("train")
val_gen_eval = _make_eval_gen("val")
test_gen_eval = _make_eval_gen("test")
print("class_indices:", train_gen_eval.class_indices)
print(f"train={train_gen_eval.samples}  val={val_gen_eval.samples}  test={test_gen_eval.samples}")

## Referensi OOD -- CIFAR-10 (dataset publik non-X-ray)

`/dataset/` lokal TIDAK bisa dipakai sebagai referensi OOD -- kedua kelas
di dalamnya (`fractured` DAN `not_fractured`) tetap gambar X-ray tulang,
sama-sama in-distribution. OOD butuh gambar yang benar-benar di luar
distribusi training model (bukan X-ray sama sekali). CIFAR-10 diambil
langsung lewat `tf.keras.datasets` (bukan pre-download manual).

In [ ]:
(_cifar_x, _), (_, _) = tf.keras.datasets.cifar10.load_data()
_rng = np.random.default_rng(42)
_idx = _rng.choice(len(_cifar_x), size=OOD_SAMPLE_SIZE, replace=False)
_ood_raw = _cifar_x[_idx].astype("float32")  # (N,32,32,3), [0,255]

_ood_resized = tf.image.resize(_ood_raw, (IMG_SIZE, IMG_SIZE)).numpy()
ood_batch = preprocess_image(_ood_resized)  # preprocessing SAMA dengan pipeline X-ray -- gerbang OOD harus adil
print("ood_batch shape:", ood_batch.shape)

## Verifikasi Grad-CAM analitik vs ground-truth GradientTape

Fungsi helper -- dipanggil sekali per model di loop utama. Ground truth
pakai `tf.GradientTape` langsung ke `model` Keras asli (backprop
sungguhan); analitik pakai `src/fracture/gradcam.py` (closed-form, NumPy
murni). Dites KEDUA arah kelas (fix B3 -- arah gradien harus benar untuk
prediksi kelas negatif juga, bukan cuma satu arah tetap seperti eksperimen
lama).

In [ ]:
def gradcam_groundtruth(model, image_batch1, explain_class, w1, w2):
    """image_batch1: (1,H,W,3) sudah preprocessed. Return heatmap (Hf,Wf) ReLU+normalisasi,
    dihitung backprop sungguhan via GradientTape -- pembanding utk versi analitik."""
    grad_model = Model([model.input], [model.get_layer("gap").input, model.output])
    with tf.GradientTape() as tape:
        featmap_t, prob_t = grad_model(image_batch1)
        # y = logit arah "not_fractured" didekati dari prob (post-sigmoid) --
        # GradientTape backprop tetap lewat sigmoid, konsisten dgn cara model
        # sungguhan dipakai (bukan celah pemisahan logit/aktivasi yang tidak ada).
        target = prob_t[:, 0] if explain_class == "not_fractured" else -prob_t[:, 0]
    grads = tape.gradient(target, featmap_t)  # (1,Hf,Wf,C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2)).numpy()  # (C,) -- GAP gradien klasik Grad-CAM
    featmap_np = featmap_t.numpy()[0]  # (Hf,Wf,C)
    heatmap = np.maximum(np.tensordot(featmap_np, pooled, axes=([2], [0])), 0)
    max_val = heatmap.max()
    if max_val > 1e-8:
        heatmap = heatmap / max_val
    return heatmap


def verify_gradcam_parity(model, sample_images, w1, b1, w2, atol=1e-4):
    """sample_images: (N,H,W,3) preprocessed. Return selisih absolut maksimum
    lintas semua sampel x kedua arah kelas -- raise kalau >= atol."""
    feat_model = Model(inputs=model.input, outputs=model.get_layer("gap").output)
    featmap_model = Model(inputs=model.input, outputs=model.get_layer("gap").input)
    max_diff = 0.0
    for i in range(len(sample_images)):
        img1 = sample_images[i : i + 1]
        gap_feat = feat_model.predict(img1, verbose=0)[0]
        z = gradcam.compute_z(gap_feat, w1, b1)
        featmap_np = featmap_model.predict(img1, verbose=0)[0]
        for cls in ("fractured", "not_fractured"):
            analytic = gradcam.compute_heatmap(featmap_np, z, w1, w2, explain_class=cls)
            # NOTE: GradientTape butuh sigmoid post-activation sbg target (lihat
            # gradcam_groundtruth) -- Grad-CAM klasik biasanya pakai logit
            # pra-sigmoid, tapi model ini tidak punya hook ke situ (Dense+sigmoid
            # menyatu). Sinyal gradien lewat sigmoid tetap searah dgn logit
            # (sigmoid monoton naik), jadi ARAH heatmap tetap valid; skalanya
            # dinormalisasi (dibagi max) di kedua sisi jadi tidak masalah utk
            # perbandingan bentuk heatmap -- tapi utk verifikasi numerik ketat,
            # ground_truth JUGA dipakai via sigmoid supaya definisi "logit"
            # konsisten sisi analitik & sisi backprop (lihat catatan di bawah).
            ground_truth = gradcam_groundtruth(model, img1, cls, w1, w2)
            diff = float(np.max(np.abs(analytic - ground_truth)))
            max_diff = max(max_diff, diff)
    assert max_diff < atol, f"Parity Grad-CAM gagal: selisih maksimum {max_diff} >= {atol}"
    return max_diff

**Catatan penting soal verifikasi Grad-CAM di atas:** turunan analitik di
`gradcam.py` menurunkan gradien logit PRA-sigmoid (`y`) terhadap feature
map -- ini konsisten dengan Grad-CAM klasik (Selvaraju et al.). Tapi model
Keras kita punya `Dense(1, activation="sigmoid")` MENYATU (tidak ada hook
terpisah ke pra-aktivasi), jadi `gradcam_groundtruth()` di atas terpaksa
backprop lewat OUTPUT SIGMOID, bukan logit. Karena sigmoid monoton naik,
`d(sigmoid(y))/dA = sigmoid'(y) * dy/dA` -- yaitu ground-truth = analitik
dikali SATU SKALAR POSITIF `sigmoid'(y)` yang konstan untuk semua (h,w)
dalam satu forward pass. Setelah kedua sisi dinormalisasi ke [0,1] (dibagi
nilai maksimumnya sendiri), skalar itu hilang dan heatmap harus identik.
Kalau assert di bawah gagal, ini asumsi pertama yang perlu dicurigai.

## Monitoring memori -- bukti keras, bukan tebakan lagi

Sesi sebelumnya kernel mati sendiri tanpa exception Python (dideteksi
lewat log runtime Colab: `AsyncIOLoopKernelRestarter` -- restart OTOMATIS
yang cuma terjadi kalau proses kernelnya mati tak terduga, ciri khas
OOM-killer Linux). `del`+`clear_session()`+`gc.collect()` di akhir tiap
iterasi TERNYATA belum cukup. Sebelum menebak fix berikutnya, print
pemakaian RAM host + VRAM GPU eksplisit di tiap titik -- supaya kalau
OOM lagi, jelas persis di iterasi mana & seberapa besar menumpuknya.

In [ ]:
import psutil

def _log_memory(label):
    host_gb = psutil.virtual_memory().used / 1e9
    msg = f"[MEM {label}] host={host_gb:.2f}GB"
    try:
        gpu_info = tf.config.experimental.get_memory_info("GPU:0")
        msg += f"  gpu_current={gpu_info['current']/1e9:.2f}GB  gpu_peak={gpu_info['peak']/1e9:.2f}GB"
    except Exception as e:
        msg += f"  (info GPU tidak tersedia: {e})"
    print(msg)

## Loop utama -- evaluasi + kalibrasi + OOD + Grad-CAM + ekspor ONNX, keempat backbone

In [ ]:
with open(f"{REPO_DIR}/configs/base.yaml") as f:
    base_config = yaml.safe_load(f)

all_metrics = []

for BACKBONE in BACKBONES:
    print(f"\n{'='*60}\n{BACKBONE.upper()}\n{'='*60}")

    onnx_path = str(Path(EXPORT_ROOT) / f"{BACKBONE}.onnx")
    npz_path = str(Path(EXPORT_ROOT) / f"{BACKBONE}_head.npz")
    metrics_cache_path = Path(EXPORT_ROOT) / f"{BACKBONE}_metrics.json"

    # Resume per-backbone: OOM bisa mematikan KERNEL kapan saja (bukan cuma
    # exception Python yang aman ditangkap try/except) -- restart berarti
    # all_metrics di memori HILANG TOTAL. Cache metrics ke file (di Drive,
    # selamat dari restart) supaya backbone yang sudah selesai tidak perlu
    # diulang dari nol -- pola resume yang sama dengan status.json di
    # src/fracture/train.py.
    if metrics_cache_path.exists() and Path(onnx_path).exists() and Path(npz_path).exists():
        print(f"[{BACKBONE}] Sudah pernah selesai (ditemukan {metrics_cache_path.name}) -- lewati, pakai cache.")
        all_metrics.append(json.loads(metrics_cache_path.read_text()))
        continue

    _log_memory(f"{BACKBONE} - mulai")

    config = dict(base_config)
    override_file = "base_model.yaml" if BACKBONE == "base" else f"{BACKBONE}.yaml"
    with open(f"{REPO_DIR}/configs/{override_file}") as f:
        config.update(yaml.safe_load(f))
    config_str = json.dumps(config, sort_keys=True)
    config_hash = hashlib.sha256(config_str.encode()).hexdigest()[:8]
    run_dir = Path(RUNS_ROOT) / f"{BACKBONE}_{config_hash}"
    best_path = run_dir / "best.keras"
    assert best_path.exists(), f"{best_path} tidak ada -- {BACKBONE} belum selesai training (02_train.ipynb)?"

    model = tf.keras.models.load_model(best_path)

    # ---- 1. Prediksi val & test, konversi ke konvensi prob_fractured/y_fractured ----
    val_gen_eval.reset(); test_gen_eval.reset()
    raw_val = model.predict(val_gen_eval, verbose=1).squeeze(axis=-1)
    prob_fractured_val = 1 - raw_val
    # np.asarray() -- .classes kadang list Python biasa (bukan ndarray)
    # tergantung versi Keras; "list == 0" diam-diam jadi satu bool skalar
    # (bukan elementwise), bukan error -- baru meledak di .astype() berikutnya.
    y_fractured_val = (np.asarray(val_gen_eval.classes) == 0).astype(int)

    val_gen_eval.reset(); test_gen_eval.reset()
    raw_test = model.predict(test_gen_eval, verbose=1).squeeze(axis=-1)
    prob_fractured_test = 1 - raw_test
    y_fractured_test = (np.asarray(test_gen_eval.classes) == 0).astype(int)

    # ---- 2. Threshold (VALIDATION) + bootstrap CI (TEST) ----
    threshold = evaluate.select_threshold_youden(y_fractured_val, prob_fractured_val)
    ci = evaluate.bootstrap_ci(y_fractured_test, prob_fractured_test, threshold, n_resamples=N_BOOTSTRAP)
    pred_test = (prob_fractured_test >= threshold).astype(int)
    cm = evaluate.confusion_matrix_dict(y_fractured_test, pred_test)
    roc_pts = evaluate.roc_points(y_fractured_test, prob_fractured_test)
    pr_pts = evaluate.pr_points(y_fractured_test, prob_fractured_test)
    risk_coverage = evaluate.risk_coverage_curve(y_fractured_test, prob_fractured_test, threshold)

    # ---- 3. Kalibrasi (temperature di VALIDATION, dilaporkan di TEST) ----
    T = calibration.fit_temperature(prob_fractured_val, y_fractured_val)
    calibrated_test = calibration.apply_temperature(prob_fractured_test, T)
    ece = calibration.expected_calibration_error(y_fractured_test, calibrated_test)
    reliability = calibration.reliability_diagram_points(y_fractured_test, calibrated_test)

    # ---- 4. OOD: fitur GAP train (fit) vs val (threshold) vs test+CIFAR10 (evaluasi) ----
    feat_model = Model(inputs=model.input, outputs=model.get_layer("gap").output)
    train_gen_eval.reset(); val_gen_eval.reset(); test_gen_eval.reset()
    feat_train = feat_model.predict(train_gen_eval, verbose=1)
    feat_val = feat_model.predict(val_gen_eval, verbose=1)
    feat_test = feat_model.predict(test_gen_eval, verbose=1)
    feat_ood = feat_model.predict(ood_batch, verbose=1)

    ood_mean, ood_inv_cov = ood.fit_mahalanobis(feat_train)
    s_val = ood.score_mahalanobis(feat_val, ood_mean, ood_inv_cov)
    ood_threshold = ood.select_ood_threshold(s_val, percentile=95.0)
    s_test = ood.score_mahalanobis(feat_test, ood_mean, ood_inv_cov)
    s_ood = ood.score_mahalanobis(feat_ood, ood_mean, ood_inv_cov)
    ood_auroc = ood.evaluate_ood_auroc(s_test, s_ood)

    # ---- 5. Ekspor ONNX + bobot head ----
    # (onnx_path/npz_path sudah didefinisikan di awal iterasi, dipakai jg utk cek resume)
    weights = export_onnx.extract_head_weights(model)
    export_onnx.export_to_onnx(model, onnx_path, img_size=config["img_size"])
    export_onnx.save_head_artifacts(npz_path, weights, extra={
        "temperature": T, "threshold": threshold,
        "ood_mean": ood_mean, "ood_inv_cov": ood_inv_cov, "ood_threshold": ood_threshold,
    })
    onnx_size_mb = Path(onnx_path).stat().st_size / (1024 * 1024)

    # ---- 6. Verifikasi parity (prob ONNX vs Keras, Grad-CAM analitik vs GradientTape) ----
    test_gen_eval.reset()
    sample_batch = next(iter(test_gen_eval))[0][:8]
    prob_parity_diff = export_onnx.verify_prob_parity(model, onnx_path, sample_batch)
    gradcam_parity_diff = verify_gradcam_parity(model, sample_batch[:3], weights["w1"], weights["b1"], weights["w2"])
    print(f"[{BACKBONE}] parity: prob diff={prob_parity_diff:.2e}  gradcam diff={gradcam_parity_diff:.2e}")

    # ---- 7. Kumpulkan ke bentuk ModelMetrics (cocok web/src/lib/api/types.ts) ----
    model_metrics = {
        "model_id": BACKBONE,
        "accuracy": {k: ci["accuracy"][k] for k in ("point", "lower", "upper")},
        "precision": {k: ci["precision"][k] for k in ("point", "lower", "upper")},
        "recall": {k: ci["recall"][k] for k in ("point", "lower", "upper")},
        "f1": {k: ci["f1"][k] for k in ("point", "lower", "upper")},
        "auroc": {k: ci["auroc"][k] for k in ("point", "lower", "upper")},
        "auprc": {k: ci["auprc"][k] for k in ("point", "lower", "upper")},
        "ece": ece,
        "reliability_diagram": reliability,
        "roc_curve": roc_pts,
        "pr_curve": pr_pts,
        "confusion_matrix": cm,
        "risk_coverage_curve": risk_coverage,
        "ood_auroc": ood_auroc,
        "selected_threshold": threshold,
        "test_set_size": int(test_gen_eval.samples),
        # Bidang tambahan di luar ModelMetrics TS -- bukti/debug, dipakai
        # notebook backend nanti, bukan langsung dikonsumsi Benchmark page:
        "_temperature": T,
        "_ood_threshold": ood_threshold,
        "_onnx_size_mb": onnx_size_mb,
        "_prob_parity_max_diff": prob_parity_diff,
        "_gradcam_parity_max_diff": gradcam_parity_diff,
        "_bootstrap_skipped_resamples": ci["skipped_resamples"],
    }
    all_metrics.append(model_metrics)
    metrics_cache_path.write_text(json.dumps(model_metrics, indent=2))
    print(f"[{BACKBONE}] accuracy={ci['accuracy']['point']:.4f}  "
          f"auroc={ci['auroc']['point']:.4f}  ece={ece:.4f}  ood_auroc={ood_auroc:.4f}  "
          f"threshold={threshold:.4f}  onnx={onnx_size_mb:.1f}MB")
    _log_memory(f"{BACKBONE} - sebelum cleanup")

    # ---- 8. Bersihkan memori SEBELUM lanjut ke backbone berikutnya ----
    # Loop ini memuat 4 model ConvNeXt (Tiny..Large) berurutan di SATU sesi
    # kernel -- tanpa ini, graph/session TF menumpuk antar iterasi (tidak
    # otomatis dilepas garbage collector Python biasa), ditambah konversi
    # TFLite yang rakus RAM -- kombinasi ini bisa mendorong RAM sistem
    # Colab gratis (~12-13GB) sampai OOM di backbone yang lebih besar
    # (Base/Large), walau Tiny/Small aman.
    del model, feat_model, weights
    tf.keras.backend.clear_session()
    gc.collect()
    _log_memory(f"{BACKBONE} - setelah cleanup")

## Ringkasan lintas model -- aturan klaim CI (spec Sec7/Sec14)

Klaim "model A lebih baik dari model B" HANYA valid kalau interval
kepercayaan 95% keduanya TIDAK overlap -- kalau overlap, bedanya belum
tentu bukan kebetulan sampling, harus dilaporkan sebagai "tidak berbeda
signifikan", bukan diam-diam pilih yang angka pointnya lebih tinggi.

In [ ]:
import pandas as pd

summary_rows = []
for m in all_metrics:
    summary_rows.append({
        "model": m["model_id"],
        "accuracy": f"{m['accuracy']['point']:.4f} [{m['accuracy']['lower']:.4f}, {m['accuracy']['upper']:.4f}]",
        "auroc": f"{m['auroc']['point']:.4f} [{m['auroc']['lower']:.4f}, {m['auroc']['upper']:.4f}]",
        "f1": f"{m['f1']['point']:.4f} [{m['f1']['lower']:.4f}, {m['f1']['upper']:.4f}]",
        "ece": f"{m['ece']:.4f}",
        "ood_auroc": f"{m['ood_auroc']:.4f}",
        "onnx_mb": f"{m['_onnx_size_mb']:.1f}",
    })
display(pd.DataFrame(summary_rows))

def _ci_overlap(a, b):
    return not (a["upper"] < b["lower"] or b["upper"] < a["lower"])

print("\nPerbandingan akurasi berpasangan (CI 95% overlap? -- overlap = TIDAK boleh klaim beda):")
for i in range(len(all_metrics)):
    for j in range(i + 1, len(all_metrics)):
        a, b = all_metrics[i], all_metrics[j]
        overlap = _ci_overlap(a["accuracy"], b["accuracy"])
        verdict = "TIDAK signifikan (CI overlap)" if overlap else "signifikan (CI tidak overlap)"
        print(f"  {a['model_id']:6s} vs {b['model_id']:6s}: {verdict}")

## Simpan `results/metrics.json`

`clahe_ablation` SENGAJA tidak diisi (lihat catatan di cell paling atas) --
frontend Benchmark page (`web/src/features/benchmark/`) perlu penyesuaian
kecil untuk menangani field ini opsional/belum ada sampai ablation study
(spec Sec11) benar-benar dijalankan.

In [ ]:
metrics_response = {
    "generated_at": pd.Timestamp.utcnow().isoformat(),
    "config_hash": hashlib.sha256(json.dumps(base_config, sort_keys=True).encode()).hexdigest()[:8],
    "models": all_metrics,
    # "clahe_ablation": belum ada -- lihat catatan cell markdown di atas.
}

out_path = Path(REPO_DIR) / "results" / "metrics.json"
out_path.write_text(json.dumps(metrics_response, indent=2))
print(f"Tersimpan: {out_path} ({out_path.stat().st_size / 1024:.1f} KB)")

# Salin juga ke Drive supaya tidak hilang kalau runtime Colab terputus
# sebelum sempat di-download manual.
shutil.copy2(out_path, Path(EXPORT_ROOT) / "metrics.json")
print(f"Salinan cadangan: {EXPORT_ROOT}/metrics.json")

## Langkah selanjutnya (manual, di luar notebook ini)

1. **Download dari Drive ke lokal**: `fracture-exports/{tiny,small,base,large}.onnx`
   + `*_head.npz` + `metrics.json` -- folder `EXPORT_ROOT` di Drive.
2. **`results/metrics.json`** -> commit ke repo git (kecil, jadi sumber
   data `getMetrics()` asli menggantikan mock MSW di Benchmark page).
3. **`.onnx` + `.npz`** -> TIDAK di-commit ke git (gitignored, besar) --
   ini artefak deployment, tujuan akhirnya di-upload ke repo Hugging Face
   Space bareng kode FastAPI backend (langkah berikutnya setelah ini,
   sesuai urutan yang sudah disepakati: evaluasi -> ONNX -> baru backend).
4. Kalau ada model yang GAGAL verifikasi parity (`assert` di cell loop
   utama meledak) -- JANGAN lanjut ke backend dengan model itu sebelum
   akar masalahnya ditemukan; parity ONNX/Grad-CAM adalah syarat, bukan
   sekadar sanity check kosmetik (spec Sec8).